# Spider Dataset Explorer & Prompt Builder
Load Spider, inspect statistics and difficulty levels, and turn each (question, schema) pair into a clean text prompt.

## 1. Load the dataset
Switch `SOURCE` to `'hf'` to pull `xlangai/spider` from Hugging Face, `'local'` for an unzipped Spider folder, or keep `'sample'` to run offline.

In [ ]:
from spider_prompt import (load_sample, load_hf, load_local, SchemaStore,
                           build_prompt, serialize_schema, LEVELS)

SOURCE = 'sample'          # 'sample' | 'hf' | 'local'
SPIDER_DIR = ''            # e.g. '/data/spider' (contains tables.json, database/, dev.json ...)

df = {'sample': load_sample, 'hf': load_hf, 'local': lambda: load_local(SPIDER_DIR)}[SOURCE]()
store = SchemaStore(f'{SPIDER_DIR}/tables.json' if SPIDER_DIR else None,
                    f'{SPIDER_DIR}/database' if SPIDER_DIR else None)
df.head()

## 2. Dataset statistics

In [ ]:
print(f'{len(df):,} examples, {df.db_id.nunique()} databases')
display(df.groupby('split').size().rename('examples').to_frame())
df[['question_words', 'query_chars']].describe().round(1)

## 3. Query difficulty levels
Heuristic re-implementation of Spider's hardness rules (easy / medium / hard / extra) applied to the raw SQL.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
counts = df.difficulty.value_counts().reindex(LEVELS, fill_value=0)
display(pd.concat([counts.rename('count'), (counts / counts.sum() * 100).round(1).rename('percent')], axis=1))
counts.plot(kind='bar', title='Difficulty distribution', rot=0); plt.show()
pd_cross = df.groupby(['split', 'difficulty']).size().unstack(fill_value=0).reindex(columns=LEVELS, fill_value=0)
pd_cross

In [ ]:
for lvl in LEVELS:
    sub = df[df.difficulty == lvl]
    if len(sub):
        r = sub.iloc[0]
        print(f'[{lvl}] {r.question}\n   {r.query}\n')

## 4. Schema serialization
Three formats are supported: `create_table`, `compact`, `markdown`.

In [ ]:
db = df.db_id.iloc[0]
schema = store.get(db)
for style in ['create_table', 'compact', 'markdown']:
    print(f'--- {style} ---')
    print(serialize_schema(schema, style), '\n')

## 5. `build_prompt(question, schema)`

In [ ]:
row = df.iloc[0]
print(build_prompt(row.question, store.get(row.db_id)))

## 6. Build prompts for the whole dataset
Skips databases with no schema available.

In [ ]:
records = []
for _, r in df.iterrows():
    s = store.get(r.db_id)
    if s is None:
        continue
    records.append({'db_id': r.db_id, 'difficulty': r.difficulty,
                    'prompt': build_prompt(r.question, s), 'gold_sql': r.query})
print(len(records), 'prompts built')
import json
with open('prompts.jsonl', 'w') as f:
    f.write('\n'.join(json.dumps(x) for x in records))